# AIM is to save document into database

#### installed : 
```bash
pip install pandas
```

In [43]:
# from django.db import connection
# from asgiref.sync import sync_to_async

In [1]:
from dotenv import load_dotenv
load_dotenv()
import psycopg2
import os
import sys

In [2]:

# import django

sys.path.append(r'D:\Rushal\VS_code\Python\ProjectIII\backend')
os.chdir(r'D:\Rushal\VS_code\Python\ProjectIII\backend')

# os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'backend.settings')
# django.setup()

# from common.models import Document_Source, InterviewQuestions

In [3]:
base_path = r"ai/documents/question_answers/"

In [4]:
all_files = os.listdir(f"{base_path}")
# backend\ai\documents

In [5]:
filepath = base_path + all_files[0]

In [6]:
import pandas as pd
import json

In [7]:
df = pd.read_excel(filepath)

In [8]:
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.width', None)

df.head(3)

,question,role,answer,roles,keywords
0,You've been tasked with implementing a new net...,Network Engineer,"That's a real situation, and I'd approach it m...",Network Engineer,"been, tasked, implementing, new, network, infr..."
1,Suppose we're planning to develop a new AI-pow...,Engineering Manager,"That's a real situation, and I'd approach it m...",Engineering Manager,"Suppose, planning, develop, new, AI-powered, a..."
2,You've been tasked to perform a penetration te...,Penetration Tester / Ethical Hacker,I'd treat this as a case of balancing competin...,Penetration Tester / Ethical Hacker,"been, tasked, perform, penetration, test, our,..."


In [9]:
df.shape

(20444, 5)

In [10]:
from ai.utils.embeddings import embed_text_local

In [19]:
em = embed_text_local("thik xau")
em[:3]

[-0.49727994203567505, 0.2624587416648865, -3.3480136394500732]

In [20]:
fake_df = df[:5]
# Step 1: Create the column as object dtype
# fake_df['question_embedding'] = None  # Temporary placeholder

# # Step 2: Convert to object dtype (critical!)
# fake_df['question_embedding'] = fake_df['question_embedding'].astype(object)
fake_df

,question,role,answer,roles,keywords
0,You've been tasked with implementing a new net...,Network Engineer,"That's a real situation, and I'd approach it m...",Network Engineer,"been, tasked, implementing, new, network, infr..."
1,Suppose we're planning to develop a new AI-pow...,Engineering Manager,"That's a real situation, and I'd approach it m...",Engineering Manager,"Suppose, planning, develop, new, AI-powered, a..."
2,You've been tasked to perform a penetration te...,Penetration Tester / Ethical Hacker,I'd treat this as a case of balancing competin...,Penetration Tester / Ethical Hacker,"been, tasked, perform, penetration, test, our,..."
3,Scenario: The DevOps team has been experiencin...,Scrum Master / Agile Coach,Good question — this is the kind of trade-off ...,Scrum Master / Agile Coach,"Scenario, DevOps, team, been, experiencing, in..."
4,In our current project to develop an autonomou...,Solutions Architect,Good question — this is the kind of trade-off ...,Solutions Architect,"our, current, project, develop, autonomous, dr..."


In [12]:
# # Step 3: Generate and assign embeddings
# for idx, row in fake_df.iterrows():
#     embedding = embed_text_local(row['question'])
#     fake_df.at[idx, 'question_embedding'] = embedding
#     print(f"✅ {idx+1}/{len(fake_df)} done")

## THIS IS HOW U SOLVE USING FROM DB for docker to USING LOCALHOST

In [13]:

# When running from Jupyter on Windows
conn = psycopg2.connect(
    host='localhost',  # ← Change this!
    port=5433,
    database=os.getenv('POSTGRES_DB', 'Vector_test'),
    user=os.getenv('POSTGRES_USER', 'Vector_Test'),
    password=os.getenv('POSTGRES_PASSWORD', 'password')
)
conn.autocommit = True
cur = conn.cursor()

#### port kaa maamla babu bhaiyaaa

In [14]:
cur.execute("SELECT current_database(), current_user, inet_server_addr(), inet_server_port();")
print(cur.fetchone())

('Vector_Database', 'postgres', '172.19.0.2', 5432)


In [15]:
cur.execute("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'
AND table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
""")

for schema, table in cur.fetchall():
    print(f"{schema}.{table}")

public.accounts_user
public.accounts_user_groups
public.accounts_user_user_permissions
public.auth_group
public.auth_group_permissions
public.auth_permission
public.common_chatmessage
public.common_document_source
public.common_interviewquestions
public.common_simpletextembedding
public.django_admin_log
public.django_content_type
public.django_migrations
public.django_session
public.interview_test_questions


In [16]:
cur.execute("SELECT datname FROM pg_database WHERE datistemplate = false;")
print(cur.fetchall())

[('postgres',), ('Vector_Database',)]


In [17]:
try:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

    cur.execute("""
        SELECT EXISTS (
            SELECT 1
            FROM pg_extension
            WHERE extname = 'vector'
        );
    """)

    if cur.fetchone()[0]:
        print("✅ pgvector extension is installed and enabled!")
    else:
        print("❌ pgvector extension was NOT found.")

except Exception as e:
    print(f"❌ Error: {e}")

✅ pgvector extension is installed and enabled!


In [51]:
for idx, rows in fake_df.iterrows():
    print(f'idx {idx}, rows : {rows[:10]}...')

idx 0, rows : question    You've been tasked with implementing a new net...
role                                         Network Engineer
answer      That's a real situation, and I'd approach it m...
roles                                        Network Engineer
keywords    been, tasked, implementing, new, network, infr...
Name: 0, dtype: str...
idx 1, rows : question    Suppose we're planning to develop a new AI-pow...
role                                      Engineering Manager
answer      That's a real situation, and I'd approach it m...
roles                                     Engineering Manager
keywords    Suppose, planning, develop, new, AI-powered, a...
Name: 1, dtype: str...
idx 2, rows : question    You've been tasked to perform a penetration te...
role                      Penetration Tester / Ethical Hacker
answer      I'd treat this as a case of balancing competin...
roles                     Penetration Tester / Ethical Hacker
keywords    been, tasked, perform, penetrati

In [18]:
try:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS interview_test_questions (
            id SERIAL PRIMARY KEY,
            question TEXT NOT NULL,
            role TEXT,
            answer TEXT,
            roles TEXT,
            keywords TEXT,
            question_embedding vector(768)
        );
    """)

    print("✅ Table created successfully!")

except Exception as e:
    print(f"❌ Error creating table: {e}")

✅ Table created successfully!


In [18]:
try:
    for _, row in fake_df.iterrows():

        cur.execute("""
            INSERT INTO interview_test_questions
            (question, role, answer, roles, keywords, question_embedding, answer_embedding)
            VALUES (%s, %s, %s, %s, %s, %s, %s);
        """, (
            row["question"],
            row["role"],
            row["answer"],
            row["roles"],
            row["keywords"],
            embed_text_local(row["question"]),
            embed_text_local(row['answer'])
        ))

    print(f"✅ Inserted {len(fake_df)} questions!")

except Exception as e:
    print(f"❌ Error inserting data: {e}")

❌ Error inserting data: name 'fake_df' is not defined


In [17]:
cur.execute("TRUNCATE TABLE interview_test_questions")

In [35]:
def semantic_search(query_embedding, table_name='interview_test_questions', top_k=1):
    # Convert list to string for PostgreSQL
    embedding_str = '[' + ','.join(map(str, query_embedding)) + ']'
    
    # Cosine similarity search
    cur.execute(f"""
        SELECT 
            question,
            answer,
            role,
            1 - (question_embedding <=> %s::vector) AS similarity
        FROM {table_name}
        WHERE question_embedding IS NOT NULL
        ORDER BY question_embedding <=> %s::vector
        LIMIT %s;
    """, (embedding_str, embedding_str, top_k))
    
    return cur.fetchall()


In [34]:
query = embed_text_local("senior")
semantic_search(query)

[("Scenario: The DevOps team has been experiencing intermittent issues with automated test failures in our Selenium Grid setup. As the Scrum Master, you've noticed that these failures mostly occur during critical business hours and have led to delays in deploying new features. Given our HIPAA compliance obligations and tight deadlines, can you walk me through your strategy for triaging this issue? Discuss your approach to identify the root cause, prioritize fixes, coordinate with the DevOps and QA teams, and ensure that such issues don't recur in the future.",
  "Good question — this is the kind of trade-off I've had to navigate before, so let me walk through how I'd handle it. I'd reproduce the issue reliably first, narrow down the root cause with logs and metrics, and resist the urge to patch symptoms, paying close attention to Selenium Grid, HIPAA compliance, intermittent issues, automated test failures since those are the constraints that actually drive the right answer here. Concr

In [ ]:
def handle_msg(userconent)
    {
        "user_content",
        "aissse"
    }